# Extension: DistilBERT vs the classical stack (Colab, **GPU**)

The competition banned neural/transformer models, so the submitted 0.826 uses TF-IDF + linear models.
This notebook measures what that constraint cost: it fine-tunes DistilBERT on the **same** 80/20 stratified split
(`random_state=42`, same row order as `src/evaluate.py --step holdout`) and reports Macro-F1, per-class metrics,
model size and latency so the accuracy/cost trade-off can be discussed with numbers.

`Runtime -> Change runtime type -> T4 GPU`. Full data, 1 epoch is roughly 20-40 min on a T4.
Set `TRAIN_N` to a smaller number for a quick trial.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/toxic_project'
DATA_DIR = f'{PROJECT}/data'
OUT_DIR = f'{PROJECT}/results/metrics'
import os; os.makedirs(OUT_DIR, exist_ok=True)

MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN, BATCH, EPOCHS, LR = 128, 32, 1, 3e-5
TRAIN_N = None        # e.g. 30000 for a quick trial; None = all training rows of the split
SEED = 42

In [ ]:
import time, json, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
torch.manual_seed(SEED); np.random.seed(SEED)
assert torch.cuda.is_available(), 'Switch the runtime to a GPU'

df = pd.read_csv(f'{DATA_DIR}/train.csv').dropna(subset=['comment']).reset_index(drop=True)   # same as evaluate.py
y = df['label'].to_numpy(); text = df['comment'].astype(str).tolist()
idx_tr, idx_va = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=SEED)
if TRAIN_N: idx_tr = idx_tr[:TRAIN_N]
print(len(idx_tr), 'train |', len(idx_va), 'val')

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4).cuda()

cnt = np.bincount(y[idx_tr], minlength=4)
w = torch.tensor(len(idx_tr) / (4 * cnt), dtype=torch.float).cuda()      # 'balanced' weights, like the classical models
loss_fn = torch.nn.CrossEntropyLoss(weight=w)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * int(np.ceil(len(idx_tr) / BATCH))
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1, anneal_strategy='linear')
scaler = torch.cuda.amp.GradScaler()

def batches(idx, bs, shuffle):
    idx = np.random.permutation(idx) if shuffle else idx
    for i in range(0, len(idx), bs):
        b = idx[i:i + bs]
        enc = tok([text[j] for j in b], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
        yield {k: v.cuda() for k, v in enc.items()}, torch.tensor(y[b]).cuda()

t0 = time.time(); step = 0
model.train()
for ep in range(EPOCHS):
    for enc, lab in batches(idx_tr, BATCH, True):
        with torch.autocast('cuda', dtype=torch.float16):
            loss = loss_fn(model(**enc).logits.float(), lab)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step(); step += 1
        if step % 200 == 0: print(f'step {step}/{steps} loss {loss.item():.4f} {time.time()-t0:.0f}s', flush=True)
train_s = time.time() - t0
print('train seconds', round(train_s))

In [ ]:
model.eval(); preds = []
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    for enc, lab in batches(idx_va, 128, False):
        preds.append(model(**enc).logits.argmax(-1).cpu().numpy())
pred = np.concatenate(preds); yva = y[idx_va]
macro = f1_score(yva, pred, average='macro')
print('Macro F1:', round(macro, 4)); print(classification_report(yva, pred, digits=4))

# bootstrap CI
rng = np.random.default_rng(0)
bs = [f1_score(yva[i], pred[i], average='macro') for i in (rng.integers(0, len(yva), len(yva)) for _ in range(500))]
ci = [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))]

# latency (GPU, single request incl. tokenisation) and size
t0 = time.time()
with torch.no_grad():
    for i in range(50):
        enc = tok([text[idx_va[i]]], truncation=True, max_length=MAX_LEN, return_tensors='pt')
        model(**{k: v.cuda() for k, v in enc.items()}).logits.argmax(-1).item()
single_ms = (time.time() - t0) * 1000 / 50
params_m = sum(p.numel() for p in model.parameters()) / 1e6

res = {'model': MODEL_NAME, 'macro_f1': float(macro), 'macro_f1_ci95': ci, 'train_seconds': round(train_s),
       'epochs': EPOCHS, 'max_len': MAX_LEN, 'train_rows': int(len(idx_tr)), 'val_rows': int(len(yva)),
       'single_request_ms_gpu': round(single_ms, 1), 'params_millions': round(params_m, 1),
       'per_class_f1': [float(v) for v in f1_score(yva, pred, average=None)]}
json.dump(res, open(f'{OUT_DIR}/distilbert.json', 'w'), indent=2); print(json.dumps(res, indent=2))

Send back `results/metrics/distilbert.json`. Compare with `holdout.json` (same split) for the accuracy / latency / size / cost trade-off.